# TCC: Detection → CWE → Fix no Colab

Ambiente de execução → Alterar tipo → **GPU**. Rode as células em ordem.

Checkpoints e `results.csv` vão para o Google Drive: o disco do Colab some quando a sessão cai.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Código
Envie o `TCC-envio.zip` quando o seletor de arquivo aparecer.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
files.upload()  # TCC-envio.zip
!unzip -oq TCC-envio.zip -d /content
%cd /content/TCC-envio

## 2. Dependências
Só o necessário para treino e métricas, nas versões do `requirements.txt`. O torch do Colab é mantido.

In [ ]:
!pip install -q transformers==5.9.0 peft==0.19.1 datasets==4.8.4 accelerate==1.13.0 safetensors==0.7.0 \
    tree-sitter==0.23.1 tree-sitter-c==0.23.4 \
    'codebleu @ git+https://github.com/k4black/codebleu@b0edb622f6a52fe9d1edc407be5061d3e1462a7f'
!python -m unittest discover -s tests

## 3. Treino + avaliação
Cada linha do `results.csv` traz a métrica do modelo **e** o baseline trivial no mesmo teste.

- `--smoke`: teste rápido (poucos passos) para ver se o ambiente funciona.
- Sem `--smoke`: treino completo. O Gemma é *gated*: aceite a licença no Hugging Face e rode `from huggingface_hub import login; login()` antes, ou tire-o de `--model`.
- Com mais VRAM, aumente `--batch-size` e reduza `--grad-accum` (o lote efetivo é o produto dos dois; o padrão é 2 × 8 = 16).

In [ ]:
OUT = '/content/drive/MyDrive/TCC/checkpoints'
!python train.py --smoke --output {OUT}

In [ ]:
!python train.py --output {OUT} --batch-size 8 --grad-accum 2 \
    --model Qwen/Qwen2.5-Coder-0.5B-Instruct NinedayWang/PolyCoder-160M openai-community/gpt2

Opcional: CWE e Fix com CVEfixes + BigVul.

In [ ]:
!python train.py --phase cwe fix --source cvefixes bigvul --output {OUT}/bigvul --batch-size 8 --grad-accum 2 \
    --model Qwen/Qwen2.5-Coder-0.5B-Instruct NinedayWang/PolyCoder-160M openai-community/gpt2

## 4. Tabela final
Modelo vs baseline. `failed` = respostas que falharam e entraram como erradas na média.

In [ ]:
import pandas as pd
from pathlib import Path

COLUMNS = {'detection': ('f1', 'baseline_all_vulnerable', 'f1'),
           'cwe': ('f1_micro', 'baseline_majority_cwe', 'f1_micro'),
           'fix': ('codebleu', 'baseline_copy_input', 'codebleu')}

def table(csv):
    rows = []
    for r in pd.read_csv(csv).query("status == 'success'").itertuples():
        m = eval(r.metrics, {'nan': float('nan')})  # dict gravado pelo próprio train.py
        metric, base_key, base_metric = COLUMNS[r.phase]
        base = m.get(base_key, {}).get(base_metric)
        rows.append({'fase': r.phase, 'modelo': r.model, 'métrica': metric, 'modelo_valor': m.get(metric),
                     'baseline': base, 'ganho': None if base is None else m.get(metric) - base,
                     'avaliados': m.get('evaluated'), 'failed': m.get('failed'), 'checkpoint': r.checkpoint})
    # Linhas antigas (sem baseline) ou repetidas: fica a mais recente de cada fase/modelo/pasta.
    return pd.DataFrame(rows).drop_duplicates(['fase', 'modelo', 'checkpoint'], keep='last').drop(columns='checkpoint')

for csv in (Path(OUT) / 'results.csv', Path(OUT) / 'bigvul/results.csv'):
    if csv.exists():
        print(csv)
        display(table(csv).round(4))